## Les before

In [1]:
%%capture
%pip install evaluate

In [2]:
! python -m pip install "comet_ml>=3.44.0" "transformers>=4.43.0" datasets torch scikit-learn accelerate

In [ ]:
%pip install unbabel-comet

  Using cached protobuf-4.25.9-cp37-abi3-manylinux2014_x86_64.whl.metadata (541 bytes)
Using cached protobuf-4.25.9-cp37-abi3-manylinux2014_x86_64.whl (295 kB)
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
      Successfully uninstalled protobuf-5.29.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.48.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
grpcio-tools 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 4.25.9 which is incompatible.
a2a-sdk 1.2.1 requires protobuf>=5.29.5, but you have protobuf 4.25.9 which is incompatible.
tensorflow 2.20.0 requires protobuf>=5.28.0, but you have protobuf 4.25.9 which is incompatible.
grain 0.2.18 requires protobuf>=5.28.3, but you have protobuf 4.25.9 which is incompatible.
wandb 0.28.1 requ

In [4]:
%pip install --force-reinstall "numpy==1.26.4" "pandas==2.2.3"

  Using cached numpy-1.26.4-cp313-cp313-linux_x86_64.whl
  Using cached pandas-2.2.3-cp313-cp313-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (89 kB)
  Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl.metadata (8.4 kB)
  Using cached pytz-2026.5-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2026.5-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached six-1.17.0-py2.py3-none-any.whl.metadata (1.7 kB)
Using cached pandas-2.2.3-cp313-cp313-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (12.7 MB)
Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl (229 kB)
Using cached pytz-2026.5-py2.py3-none-any.whl (506 kB)
Using cached tzdata-2026.5-py2.py3-none-any.whl (347 kB)
Using cached six-1.17.0-py2.py3-none-any.whl (11 kB)
  Attempting uninstall: pytz
    Found existing installation: pytz 2026.5
    Uninstalling pytz-2026.5:
      Successfully uninstalled pytz-2026.5
  Attempting uninstall: tzdata
    Found existing installation: tzdata 2026.5
   

In [5]:
%pip install --upgrade "protobuf>=5.28.0,<6"

  Using cached protobuf-5.29.6-cp38-abi3-manylinux2014_x86_64.whl.metadata (592 bytes)
Using cached protobuf-5.29.6-cp38-abi3-manylinux2014_x86_64.whl (320 kB)
  Attempting uninstall: protobuf
    Found existing installation: protobuf 4.25.9
    Uninstalling protobuf-4.25.9:
      Successfully uninstalled protobuf-4.25.9
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.48.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
unbabel-comet 2.2.7 requires protobuf<5.0.0,>=4.24.4, but you have protobuf 5.29.6 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [6]:
import os
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"

In [ ]:
# Les importations
import evaluate
import matplotlib.pyplot as plt
import numpy as np
#import tensorflow as tf
import torch
from datasets import Dataset, DatasetDict, load_dataset
#from comet import download_model, load_from_checkpoint
from torch.cuda.amp import GradScaler, autocast
from transformers import (
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    TrainerCallback,
    get_linear_schedule_with_warmup,
)

SOURCE_LANG_CODE_NLLB =  "fra_Latn" 
TARGET_LANG_CODE_NLLB =  "arb_Latn" 
max_input_length = 128
max_target_length = 128


class MemoryCleanupCallback(TrainerCallback):
    def on_step_end(self, args, state, control, **kwargs):
        torch.cuda.empty_cache()  # Libérer la mémoire GPU inutilisée

def plot_loss_curves(trainer, model_name) :
    log_history = trainer.state.log_history

    # Extract training / validation loss
    train_losses = [log["loss"] for log in log_history if "loss" in log]
    epoch_train = [log["step"] for log in log_history if "loss" in log]
    eval_losses = [log["eval_loss"] for log in log_history if "eval_loss" in log]
    epoch_eval = [log["step"] for log in log_history if "eval_loss" in log]

    # Plot the training loss
    plt.plot(epoch_train, train_losses, label="Training Loss")
    plt.plot(epoch_eval, eval_losses, label="Validation Loss")
    plt.xlabel("Step")
    plt.ylabel("Loss")
    plt.title(f"Training vs Validation Loss({model_name})")
    plt.legend()
    plt.grid(True)
    plt.show()


def training (modele_name, train_test_val_dataset) : 
    
    model = AutoModelForSeq2SeqLM.from_pretrained(modele_name)
    tokenizer = AutoTokenizer.from_pretrained(modele_name)

    def postprocess_text(preds, labels):
        preds = [pred.strip() for pred in preds]
        labels = [[label.strip()] for label in labels]  # ROUGE/METEOR attend des listes imbriquées
        return preds, labels


    # Charger les métriques une seule fois, en dehors de la fonction
    bleu_metric = evaluate.load("sacrebleu")
    rouge_metric = evaluate.load("rouge")
    meteor_metric = evaluate.load("meteor")
    chrf_metric = evaluate.load("chrf")
    comet_metric = evaluate.load("comet")

    def compute_metrics(eval_preds):
        preds, labels = eval_preds

        if isinstance(preds, tuple):
            preds = preds[0]

        # Remplacer les valeurs ignorées par le token PAD
        preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
        labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

        # Décodage des prédictions et des références
        decoded_preds = tokenizer.batch_decode(
            preds, skip_special_tokens=True
        )
        decoded_labels = tokenizer.batch_decode(
            labels, skip_special_tokens=True
        )

        # Nettoyage du texte
        decoded_preds, decoded_labels = postprocess_text(
            decoded_preds, decoded_labels
        )

        # Calcul des métriques
        bleu_result = bleu_metric.compute(
            predictions=decoded_preds,
            references=decoded_labels
        )

        rouge_result = rouge_metric.compute(
            predictions=decoded_preds,
            references=decoded_labels
        )

        meteor_result = meteor_metric.compute(
            predictions=decoded_preds,
            references=decoded_labels
        )

        chrf_result = chrf_metric.compute(
            predictions=decoded_preds,
            references=decoded_labels,
            word_order=2  # chrF++ : caractères + n-grammes de mots
        )

        # COMET nécessite les sources en plus des traductions et références.
        # Voir l'explication ci-dessous pour récupérer les sources.
        comet_result = comet_metric.compute(
            predictions=decoded_preds,
            references=decoded_labels
        )

        result = {
            "bleu": bleu_result["score"],
            "rouge": rouge_result["rougeL"],
            "meteor": meteor_result["meteor"],
            "chrf++": chrf_result["score"],
            "comet": comet_result["mean_score"],
        }

        return {k: round(float(v), 4) for k, v in result.items()}


    def extract_langage(examples) :
        # On va extraire le langage arabe et le langage français
        inputs = [fr['fr'] for fr in examples["translation"]]
        target = [arb['arb'] for arb in examples["translation"]]
        return {'inputs' : inputs, 'target' : target}



    def preprocess_function(examples):
        tokenizer.src_lang = SOURCE_LANG_CODE_NLLB
        model_inputs = tokenizer(
            examples["inputs"],
            max_length=max_input_length,
            truncation=True
        )

        # Nettoyer les cibles : enlever None ou chaînes vides
        clean_targets = []
        for t in examples["target"]:
            if isinstance(t, str) and t.strip():
                clean_targets.append(t)
            else:
                clean_targets.append(" ")

        tokenizer.tgt_lang = TARGET_LANG_CODE_NLLB
        labels = tokenizer(
            clean_targets,
            max_length=max_target_length,
            truncation=True,
            text_target=clean_targets  # Pour compatibilité future
        )

        model_inputs["labels"] = labels["input_ids"]
        
        return model_inputs



    # On va appliquer la fonction sur notre dataset
    train_dataset = train_test_val_dataset.map(extract_langage, batched=True, remove_columns=["translation"])
    # On va appliquer la fonction sur notre dataset
    train_dataset = train_dataset.map(preprocess_function, batched=True, remove_columns=["inputs", "target"])
    # Dataloader
    data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, label_pad_token_id=-100)


    early =  EarlyStoppingCallback(early_stopping_patience = 5)


    training_args = Seq2SeqTrainingArguments(
        output_dir=f"../src/{modele_name}",
        report_to="none",
        #hub_model_id= os.environ.get('HUB_ID_2'),  
        eval_strategy="steps",
        eval_steps=5,              
        logging_steps=5, 
        learning_rate=2e-5,
        #learning_rate=1e-5,
        gradient_checkpointing=True,  
        #use_cache = False,
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        weight_decay=0.01,
        save_total_limit=3,
        num_train_epochs=50,
        metric_for_best_model="eval_loss",
        predict_with_generate=True,
        fp16=True, 
        #push_to_hub=True,
        remove_unused_columns=False,
        load_best_model_at_end=True,
        save_strategy="steps",
        #save_steps = 50,
        #resume_from_checkpoint=True,
        gradient_accumulation_steps=4,

    )




    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset["train"],
        eval_dataset=train_dataset["val"],
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[early, MemoryCleanupCallback], #CheckpointCleanupCallback()
    )
    torch.cuda.empty_cache()  # Libérer la mémoire GPU inutilisée

    trainer.train()

    results = trainer.evaluate(train_dataset["test"]) 

    return trainer,results




#### Data splitting

In [8]:
from datasets import load_dataset,Dataset, DatasetDict, concatenate_datasets

def dataset_loader(path) :
    with open(path, "r") as file:
        data = file.readlines()

    data_clean = [line for line in data if line.strip()]

    return data_clean


In [9]:
PATH = "/kaggle/input/arb-fr-2/"
data_arb_bible = dataset_loader(PATH + "arabe_tchad3.txt")[:20]
data_fr_bible = dataset_loader(PATH + "V_francais_3.txt")[:20]

# ## Ceux non biblique
# data_arb_not_bible = dataset_loader(PATH + "arabic_not_bible")
# data_fr_not_bible = dataset_loader(PATH + "french_not_bible")

In [10]:
# Création du dictionnaire
dataset_dict_bible = {"translation": [{"arb": a, "fr": f} for a, f in zip(data_arb_bible, data_fr_bible)]}
#dataset_dict_not_bible = {"translation": [{"arb": a, "fr": f} for a, f in zip(data_arb_not_bible, data_fr_not_bible)]}


In [11]:
def equal_split(data_dict1: dict, data_dict2: dict | None = None) -> DatasetDict:
    def split_dataset(data_dict: dict) -> DatasetDict:
        dataset = Dataset.from_dict(data_dict)
        train_test = dataset.train_test_split(
            test_size=0.2,
            seed=42,
            shuffle=True,
        )
        val_test = train_test["test"].train_test_split(
            test_size=0.5,
            seed=42,
            shuffle=True,
        )

        return DatasetDict({
            "train": train_test["train"],
            "val": val_test["train"],
            "test": val_test["test"],
        })

    datasets = [split_dataset(data_dict1)]

    if data_dict2 is not None:
        datasets.append(split_dataset(data_dict2))

    return DatasetDict({
        split: concatenate_datasets([dataset[split] for dataset in datasets])
        if len(datasets) > 1
        else datasets[0][split]
        for split in ("train", "val", "test")
    })


In [12]:
train_test_val_dataset = equal_split(dataset_dict_bible)
train_test_val_dataset

DatasetDict({
    train: Dataset({
        features: ['translation'],
        num_rows: 16
    })
    val: Dataset({
        features: ['translation'],
        num_rows: 2
    })
    test: Dataset({
        features: ['translation'],
        num_rows: 2
    })
})

In [13]:
train_test_val_dataset.save_to_disk("../Data/train_test_split")

Saving the dataset (0/1 shards):   0%|          | 0/16 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2 [00:00<?, ? examples/s]

## Les after

In [14]:
%pip install evaluate
%pip install sacrebleu

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [15]:
import os
import sys
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)


In [16]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [17]:
#from src.models import training
from datasets import load_from_disk
from transformers import get_linear_schedule_with_warmup, AutoTokenizer,TrainerCallback,AutoModelForCausalLM, DataCollatorForSeq2Seq,AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer,EarlyStoppingCallback

### loading le train,test, val et model et tokenizer

In [18]:
train_test_val_dataset = load_from_disk("../Data/train_test_split")

### loading the model et tokenizer

In [19]:
nllb = "facebook/nllb-200-distilled-600M"
# model = AutoModelForSeq2SeqLM.from_pretrained(modele_name)
# tokenizer = AutoTokenizer.from_pretrained(modele_name)

In [20]:
trainer_nllb, nllb_evaluation = training(nllb, train_test_val_dataset)

trainer_nllb

[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
/usr/local/lib/python3.13/dist-packages/torchmetrics/utilities/imports.py:23: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.6. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../root/.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
/usr/local/lib/python3.13/dist-packages/pytorch_lightning/core/saving.py:216: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']


Map:   0%|          | 0/2 [00:00<?, ? examples/s]

`use_cache=True` is incompatible with gradient checkpointing`. Setting `use_cache=False`...
/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss,Validation Loss


OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 4.81 MiB is free. Including non-PyTorch memory, this process has 14.55 GiB memory in use. Of the allocated memory 14.10 GiB is allocated by PyTorch, and 257.83 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
plot_loss_curves(trainer_nllb, nllb)

## ByT5

### ByT5 : Le Modèle Textuel au Niveau de l'Octet

**ByT5** est un modèle d’intelligence artificielle textuel développé par **Google en 2021**. Sa particularité unique est de **lire et écrire le texte au niveau des octets (bytes)**, au lieu d’utiliser des mots ou des morceaux de mots (*tokens*) comme le font ChatGPT ou le T5 standard.

### 3 Caractéristiques Fondamentales

*   **Zéro Tokenizer** : Les modèles classiques découpent les phrases en jetons (ex: `"ordinateur"` devient `ordin` + `ateur`). Si un mot est rare ou mal orthographié, le modèle bugge (erreur `<unk>`). ByT5 convertit directement le texte en UTF-8 : chaque caractère ou accent est traité comme un nombre entre 0 et 255.
*   **Insensible aux fautes et au bruit** : Comme il analyse le texte "lettre par lettre", il est extrêmement robuste. Qu'on lui écrive `"ordinateur"`, `"oridnaetru"` (faute de frappe) ou `"💻"`, il comprend le contexte là où les autres modèles échouent.
*   **Architecture Text-to-Text** : Il utilise une structure *Encoder-Decoder* (comme le modèle T5 de base). Il prend n'importe quel texte en entrée et génère un autre texte en sortie, ce qui le rend idéal pour la traduction, la correction d'orthographe ou le résumé.


In [ ]:
byt5 = "google/byt5-small"

trainer_byt5, evaluation_byt5 = training(byt5, train_test_val_dataset)


In [ ]:
plot_loss_curves(trainer_byt5, byt5)

# Afribyt5

### AfriByT5 : Le Modèle Dédié aux Langues Africaines

**AfriByT5** est un modèle d’intelligence artificielle textuel développé en **2022 par la communauté africaine Masakhane NLP**. Il est spécialement conçu pour comprendre et traiter efficacement les langues africaines.

### 3 Piliers Essentiels

*   **Basé sur les octets (Byte-level)** : Tout comme le modèle ByT5 de Google dont il hérite, AfriByT5 n'utilise pas de découpage en mots ou morceaux de mots (*pas de tokenizer*). Il lit le texte caractère par caractère sous forme d'octets. Cela lui permet de gérer nativement les accents particuliers, les caractères spéciaux et les variations d'écriture sans générer d'erreurs de mots inconnus (`<unk>`).
*   **Spécialisé sur l'Afrique (Continual Pre-training)** : Bien que ByT5 soit global, il manquait de précision sur le continent africain. L'équipe de Masakhane a ré-entraîné ce modèle de manière intensive sur un corpus de **12 Go de textes africains**. Il est ainsi devenu expert dans **17 à 18 langues clés** (telles que le *Swahili*, le *Yoruba*, l'*Amharique*, le *Zulu*, l'*Haoussa*, etc.).
*   **Architecture Text-to-Text robuste** : Il fonctionne selon une structure *Encoder-Decoder*. Vous lui donnez un texte en entrée, il génère un texte en sortie. Sa lecture "lettre par lettre" le rend extrêmement résistant au bruit du web, aux fautes de frappe courantes et aux mélanges de langues (*code-switching*).


In [ ]:
model_afri = "masakhane/afri-byt5-base"

trainer_afri, evaluation_afri = training(model_afri, train_test_val_dataset)

In [ ]:
plot_loss_curves(trainer_afri, model_afri)